# RC9.2.1 (A34-A38) — 7 scenarios at 14400s + release gates
## Variant A: **no Google Drive**

Everything stays in the Colab VM and you download a results ZIP at the end.
Simplest to run; **if the VM disconnects the results are lost**, so prefer more
shards over long single runs.

Upload `RC9_2_1_COLAB_GATE_PACKAGE_A34_A38.zip` when the upload cell asks.

## Running the 7 scenarios in parallel

Each scenario is independent, so open **several Colab instances** and give each a
different shard. With 4 instances:

| instance | set this |
|---|---|
| 1 | `SHARD = 0` |
| 2 | `SHARD = 1` |
| 3 | `SHARD = 2` |
| 4 | `SHARD = 3` |

`SHARDS` must be the same number in all of them. Or name scenarios directly with
`ONLY = "CRICUT_VOICE,NMG_SP"`.

Budget is **14400s (4h) per scenario** - the DEEP design point. Run ONE scenario per instance. At 2700s the Stage-1 portfolio is still truncated and gates 2/9 report NOT_COMPARABLE_SEARCH_TRUNCATED, so a short run cannot answer the RC9.1 comparison. One instance running two scenarios takes
about 90 minutes plus overhead. Colab free tier disconnects after roughly 90
minutes idle — keep the tab open, and prefer more shards over more scenarios per
shard.

**Runtime type:** CPU is fine. This is a CP-SAT solver; a GPU does nothing for
it. Pick a **High-RAM** runtime if offered.

In [ ]:
#@title 1. Environment — pin the solver version { display-mode: "form" }
# ortools 9.15.6755 is the version every RC9.2.1 result so far was produced on.
# A different solver build can change candidate selection, so pin it.
!pip -q install "ortools==9.15.6755" "openpyxl>=3.1" 2>&1 | tail -2

import multiprocessing, platform, sys
print("python :", platform.python_version())
print("cpus   :", multiprocessing.cpu_count())
import ortools; print("ortools:", ortools.__version__)

In [ ]:
#@title 2. Upload the package ZIP { display-mode: "form" }
from google.colab import files
import pathlib, zipfile, io, os

os.makedirs("/content/rc921", exist_ok=True)
uploaded = files.upload()          # choose RC9_2_1_COLAB_GATE_PACKAGE_A34_A38.zip
name = next(iter(uploaded))
with zipfile.ZipFile(io.BytesIO(uploaded[name])) as z:
    z.extractall("/content/rc921")
print("extracted to /content/rc921")
!ls /content/rc921

In [ ]:
#@title 3. Locate the package { display-mode: "form" }
import pathlib
cands = list(pathlib.Path("/content/rc921").rglob("SCENARIOS.json"))
assert cands, "SCENARIOS.json not found - did the ZIP extract correctly?"
PACKAGE_ROOT = str(cands[0].parent)
RESULTS_ROOT = pathlib.Path(PACKAGE_ROOT) / "results"
RESULTS_ROOT.mkdir(exist_ok=True)
print("PACKAGE_ROOT =", PACKAGE_ROOT)
print("RESULTS_ROOT =", RESULTS_ROOT)

In [ ]:
#@title 4. Verify the package before running anything { display-mode: "form" }
# The guard suite is offline and takes seconds. If the engine in this package
# does not match its own tests, nothing produced afterwards is worth comparing.
import subprocess, sys, json, pathlib
ROOT = pathlib.Path(PACKAGE_ROOT)

manifest = json.loads((ROOT / "SCENARIOS.json").read_text())
print("package:", manifest["package"])
print("engine :", manifest["engine_release"])
print("sha256 :", manifest["engine_sha256"][:32], "…")
print()
for row in manifest["scenarios"]:
    print(f"  {row['scenario_id']:14} {row['input']}")
print()

proc = subprocess.run([sys.executable, str(ROOT / "runners" / "rc921_runner.py"),
                       "--package-root", str(ROOT), "--gate-only",
                       "--results-root", str(ROOT / "results")],
                      capture_output=True, text=True)
print("(no results yet is expected on a fresh package)")

In [ ]:
#@title 5. Run the scenarios { display-mode: "form" }
SHARD  = 0    #@param {type:"integer"}
SHARDS = 1    #@param {type:"integer"}
ONLY   = ""   #@param {type:"string"}
TIME_LIMIT = 14400  #@param {type:"integer"}
SKIP_GUARDS = False #@param {type:"boolean"}

import subprocess, sys, multiprocessing
cmd = [sys.executable, "-u", str(ROOT / "runners" / "rc921_runner.py"),
       "--package-root", str(ROOT),
       "--results-root", str(RESULTS_ROOT),
       "--shard", str(SHARD), "--shards", str(SHARDS),
       "--time-limit", str(TIME_LIMIT),
       "--num-workers", str(multiprocessing.cpu_count())]
if ONLY.strip():  cmd += ["--only", ONLY.strip()]
if SKIP_GUARDS:   cmd += ["--skip-guards"]

print(" ".join(cmd), "\n")
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
print("\nexit code:", proc.wait())

In [ ]:
#@title 6. Score the gates (re-runnable on its own) { display-mode: "form" }
import subprocess, sys
proc = subprocess.run([sys.executable, str(ROOT / "tools" / "release_gate_report.py"),
                       str(RESULTS_ROOT), "--out-dir", str(RESULTS_ROOT / "_gate_report")],
                      capture_output=True, text=True)
print(proc.stdout)
if proc.returncode: print(proc.stderr[-3000:])

## 7. Download the results

Send back the **whole** ZIP. It contains `RUN_LEDGER.json` (which scenarios ran,
on what hardware, wall clock vs budget) and `_gate_report/` — both are needed to
interpret the numbers.

In [ ]:
#@title 7. Package and download results { display-mode: "form" }
import shutil, time
from google.colab import files
stamp = time.strftime("%Y%m%d_%H%M%S")
out = f"/content/RC921_RESULTS_shard{SHARD}of{SHARDS}_{stamp}"
shutil.make_archive(out, "zip", str(RESULTS_ROOT))
print("size:", round(os.path.getsize(out + ".zip")/1e6, 1), "MB")
files.download(out + ".zip")